In [ ]:
# =============================================================================
# TASK 1: Graph Concepts & State Design
# =============================================================================
# LangGraph builds agents as GRAPHS instead of flat loops.
#
# CORE CONCEPTS:
# - StateGraph: Container for your workflow (like a flowchart)
# - State: Shared dictionary that flows through all nodes
# - Node: A function that reads state, does work, returns updated state
# - Edge: Transition to the next node
# - Conditional Edge: "If X, go here; if Y, go there"
#
# Today we build a Research Assistant that can RETRY if its answer is bad.
# =============================================================================

from typing import TypedDict, Annotated
import operator

# --- STATE SCHEMA ---
# This defines what data flows through the graph.
# Every node reads from this and writes to this.
class ResearchState(TypedDict):
    query: str                          # What the user asked
    plan: str                           # What the agent plans to do
    search_results: str                 # Raw results from tools
    answer: str                         # Generated answer
    quality_score: int                  # Score 0-10 (10 = perfect)
    retry_count: int                    # How many times we retried
    max_retries: int                    # Safety limit

# --- NODES ---
# Each node is a function: takes state, returns updated fields.

def plan_node(state):
    """Node 1: Create a plan based on the query."""
    query = state["query"]
    plan = f"Search for '{query}', then summarize findings"
    print(f"  [PLAN] Created plan: {plan}")
    return {"plan": plan}

def execute_node(state):
    """Node 2: Execute the plan (simulate search)."""
    plan = state["plan"]
    # Simulating search results (in real app, call your tools here)
    results = f"Found info about: {state['query']}. Key facts: LangGraph is a graph-based agent framework."
    print(f"  [EXECUTE] Got results: {results[:50]}...")
    return {"search_results": results}

def generate_node(state):
    """Node 3: Generate an answer from search results."""
    results = state["search_results"]
    # Simulating answer generation (in real app, call LLM here)
    answer = f"Based on research: {results}. LangGraph gives you branching and self-correction loops."
    print(f"  [GENERATE] Generated answer: {answer[:50]}...")
    return {"answer": answer}

def critique_node(state):
    """Node 4: Critique the answer and assign a quality score."""
    answer = state["answer"]
    retry = state["retry_count"]
    
    # Simulating quality check (in real app, call LLM to evaluate)
    # First attempt usually gets lower score, retry improves it
    if retry == 0:
        score = 5  # Below threshold -> will retry
    else:
        score = 8  # Above threshold -> done
    
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {"quality_score": score}

# --- CONDITIONAL ROUTING ---
# This function decides where to go next based on state.
# This is the "if score >= 7, finish; else retry" logic.

def route_after_critique(state):
    """Decide: finish or retry?"""
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    
    if score >= 7:
        print(f"  [ROUTE] Score {score} >= 7 -> FINISH")
        return "finish"
    elif retries < max_retries:
        print(f"  [ROUTE] Score {score} < 7 and retries {retries} < {max_retries} -> RETRY")
        return "retry"
    else:
        print(f"  [ROUTE] Max retries reached -> FINISH anyway")
        return "finish"

# --- TEST THE LOGIC (without building the full graph yet) ---
# This simulates what the graph would do, proving the logic works.

print("=" * 60)
print("SIMULATING THE GRAPH (before building it)")
print("=" * 60)

# Initial state
state: ResearchState = {
    "query": "What is LangGraph?",
    "plan": "",
    "search_results": "",
    "answer": "",
    "quality_score": 0,
    "retry_count": 0,
    "max_retries": 3,
}

# Run the nodes in order
print("\n--- Pass 1 ---")
state.update(plan_node(state))
state.update(execute_node(state))
state.update(generate_node(state))
state.update(critique_node(state))

# Check routing
next_step = route_after_critique(state)

# If retry, run again
if next_step == "retry":
    state["retry_count"] += 1
    print(f"\n--- Pass 2 (retry #{state['retry_count']}) ---")
    state.update(execute_node(state))  # Re-execute
    state.update(generate_node(state))  # Re-generate
    state.update(critique_node(state))  # Re-critique
    next_step = route_after_critique(state)

print("\n" + "=" * 60)
print("FINAL STATE")
print("=" * 60)
print(f"  Query: {state['query']}")
print(f"  Answer: {state['answer'][:60]}...")
print(f"  Quality: {state['quality_score']}/10")
print(f"  Retries: {state['retry_count']}")
print(f"  Decision: {next_step}")


+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|                                                                    |
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+  